# Homework 10 - Maps (optional)
In this homework, we want to see a map visualization. Feel free to use any dataset you like. We recommend the use of Altair.

## Instructions

1. **Project Setup**:  
   - Set up your Python and Jupyter (or VSCode) environment.  
   - Clone or download the repository provided in class (refer to the class notes).

2. **Choose a Dataset**:  
   - You can use any dataset with geographic data. Even a dataset you have used for your project is fine.

3. **Identify The Geolocated Features**:
   - Identify which features can be used for geolocation in your dataset. Examples include latitude and longitude, city names, state names, country names, or any other geographic identifiers.

4. **Choose a map visualization approach**:
   - Select a map visualization based on the methods explored in class or in the [previous materials](../class).
   - We recommend using Altair for its simplicity and effectiveness in creating interactive visualizations. But you are free to use other libraries.

5. **Documentation**:  
   - Comment your code and add markdown explanations for each part of your analysis.

6. **Example datasets**
 - **Global Earthquakes - USGS** (CSV feeds)
   [https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php)

 - **Airbnb Listings - Inside Airbnb** (CSV per city)
   [http://insideairbnb.com/get-the-data/](http://insideairbnb.com/get-the-data/)

 - **Meteorite Landings - NASA Open Data** (CSV)
   https://www.kaggle.com/datasets/nasa/meteorite-landings

 - **US County Population (2018 estimates) - Census Bureau** (CSV)
    [https://www2.census.gov/programs-surveys/popest/datasets/2010-2018/counties/totals/co-est2018-alldata.csv](https://www2.census.gov/programs-surveys/popest/datasets/2010-2018/counties/totals/co-est2018-alldata.csv)
    
 - **World Happiness Report (by country)** (CSV)
    https://worldhappiness.report/data-sharing/


7. **Submission**:  
   - Ensure your notebook is complete and all cells are executed without errors.
   - Save your notebook and export as either PDF or HTML. If the visualizations using altair are not being shown in the html, submit a separated version with altair html. Refer to: https://altair-viz.github.io/getting_started/starting.html#publishing-your-visualization (you can use the `chart.save('chart_file.html')` method).
   - Submit to Canvas.

In [1]:
# Install required packages
!pip install altair vega_datasets

# Import necessary libraries
import pandas as pd
import altair as alt

# Load the earthquake data
df = pd.read_csv('/content/all_month.csv')  # Adjust path if needed

# Display sample
df.head()


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-04-30T04:58:06.380Z,38.829498,-122.835831,10.610,0.72,md,9.0,187.0,0.006404,0.04,...,2025-04-30T04:59:41.149Z,"9 km NW of The Geysers, CA",earthquake,1.81,7.950000,0.310,10.0,automatic,nc,nc
1,2025-04-30T04:44:36.255Z,31.568000,-104.011000,5.915,1.90,ml,16.0,58.0,0.000000,0.20,...,2025-04-30T04:50:50.768Z,"34 km NW of Toyah, Texas",earthquake,0.00,2.561422,0.200,12.0,automatic,tx,tx
2,2025-04-30T04:43:25.550Z,33.300000,-116.156167,2.920,0.86,ml,44.0,42.0,0.134800,0.24,...,2025-04-30T04:46:59.003Z,"18 km N of Ocotillo Wells, CA",earthquake,0.23,0.540000,0.218,21.0,automatic,ci,ci
3,2025-04-30T04:24:10.510Z,33.532500,-116.722667,5.370,0.15,ml,16.0,70.0,0.035020,0.15,...,2025-04-30T04:28:32.957Z,"5 km WSW of Anza, CA",earthquake,0.25,0.530000,0.090,7.0,automatic,ci,ci
4,2025-04-30T03:58:52.633Z,59.714800,-152.728800,87.200,4.00,ml,NaN,NaN,NaN,0.59,...,2025-04-30T05:07:38.262Z,"50 km W of Anchor Point, Alaska",earthquake,NaN,0.200000,NaN,NaN,reviewed,ak,ak


In [2]:
# Select relevant columns
df = df[['time', 'latitude', 'longitude', 'depth', 'mag', 'place', 'type']]

# Drop rows with missing coordinates or magnitude
df = df.dropna(subset=['latitude', 'longitude', 'mag'])

# Convert time to datetime
df['time'] = pd.to_datetime(df['time'])

# Preview data
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10201 entries, 0 to 10200
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   time       10201 non-null  datetime64[ns, UTC]
 1   latitude   10201 non-null  float64            
 2   longitude  10201 non-null  float64            
 3   depth      10201 non-null  float64            
 4   mag        10201 non-null  float64            
 5   place      10201 non-null  object             
 6   type       10201 non-null  object             
dtypes: datetime64[ns, UTC](1), float64(4), object(2)
memory usage: 558.0+ KB


In [4]:
# Enable VegaFusion to support more than 5000 rows
import altair as alt
alt.data_transformers.enable('vegafusion')


DataTransformerRegistry.enable('vegafusion')

In [7]:
# Install vegafusion with the embed feature
!pip install "vegafusion[embed]>=1.5.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.6 MB/s eta 0:00:00


In [11]:
import altair as alt

# Trim to 5000 rows (Altair's default limit)
df_small = df.head(5000)

# Basic chart without vegafusion
quake_map = alt.Chart(df_small).mark_circle(opacity=0.6).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.Size('mag:Q', scale=alt.Scale(range=[10, 1000]), title='Magnitude'),
    color=alt.Color('depth:Q', scale=alt.Scale(scheme='turbo'), title='Depth (km)'),
    tooltip=[
        alt.Tooltip('place:N', title='Location'),
        alt.Tooltip('mag:Q', title='Magnitude'),
        alt.Tooltip('depth:Q', title='Depth (km)'),
        alt.Tooltip('time:T', title='Time')
    ]
).properties(
    width=800,
    height=400,
    title='Global Earthquakes - Last 30 Days'
).project(
    type='naturalEarth1'
)

quake_map


ImportError: The vl-convert Vega-Lite compiler and file export feature requires
version 1.6.0 or greater of the 'vl-convert-python' package. 
This can be installed with pip using:
   pip install "vl-convert-python>=1.6.0"
or conda:
   conda install -c conda-forge "vl-convert-python>=1.6.0"

ImportError: vl-convert-python

alt.Chart(...)